In [56]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [57]:
output_dir = Path("../output")
datafiles = sorted(output_dir.glob("*/*/experiment_data.*"))
print(f"Found {len(datafiles)} data files:\n")
for f in datafiles:
    task_dir = f.parent.parent.name
    run_dir = f.parent.name
    size_kb = f.stat().st_size / 1024
    print(f"  {task_dir}/{run_dir}/{f.name}  ({size_kb:.1f} KB)")

Found 120 data files:

  stop_signal_task_(rdoc)/2026-03-01_12-54-37/experiment_data.csv  (203.8 KB)
  stop_signal_task_(rdoc)/2026-03-01_12-54-39/experiment_data.csv  (194.0 KB)
  stop_signal_task_(rdoc)/2026-03-01_12-54-41/experiment_data.csv  (194.0 KB)
  stop_signal_task_(rdoc)/2026-03-01_12-54-43/experiment_data.csv  (183.5 KB)
  stop_signal_task_(rdoc)/2026-03-01_12-54-45/experiment_data.csv  (183.6 KB)
  stop_signal_task_(rdoc)/2026-03-01_14-48-29/experiment_data.csv  (183.4 KB)
  stop_signal_task_(rdoc)/2026-03-01_14-48-31/experiment_data.csv  (183.4 KB)
  stop_signal_task_(rdoc)/2026-03-01_14-48-33/experiment_data.csv  (183.5 KB)
  stop_signal_task_(rdoc)/2026-03-01_14-48-35/experiment_data.csv  (183.5 KB)
  stop_signal_task_(rdoc)/2026-03-01_14-48-37/experiment_data.csv  (183.5 KB)
  stop_signal_task_(rdoc)/2026-03-01_15-10-41/experiment_data.csv  (183.5 KB)
  stop_signal_task_(rdoc)/2026-03-01_15-10-43/experiment_data.csv  (193.9 KB)
  stop_signal_task_(rdoc)/2026-03-01_15-1

In [58]:
def detect_format(df: pd.DataFrame) -> str:
    """Detect experiment data format from column names."""
    cols = set(df.columns)
    if "signal" in cols and "SSD" in cols:
        return "stopit"
    if "trial_id" in cols and "SS_trial_type" in cols:
        return "expfactory_stop_signal"
    if "trial_id" in cols and "stim_color" in cols:
        return "expfactory_stroop"
    if "text" in cols and "colour" in cols:
        return "cognitionrun_stroop"
    if "trial_id" in cols:
        return "expfactory_generic"
    return "unknown"


def analyze_expfactory_stroop(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stroop data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")

    results = {"task": "Stroop (ExpFactory)", "format": "expfactory_stroop", "n_total": len(test)}
    summary = {}
    for cond in ["congruent", "incongruent"]:
        g = test[test["condition"] == cond]
        responded = g[g["rt"].notna()]
        correct = g[g["correct_trial"] == 1]
        rt_correct = correct["rt"].dropna()
        summary[f"{cond}_accuracy"] = g["correct_trial"].mean() if len(g) else np.nan
        summary[f"{cond}_omission_rate"] = g["rt"].isna().mean() if len(g) else np.nan
        summary[f"{cond}_rt"] = rt_correct.mean() if len(rt_correct) else np.nan
    results["summary"] = summary
    return results


def analyze_expfactory_stop_signal(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stop Signal data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")
    test["SSD"] = pd.to_numeric(test.get("SSD", pd.Series(dtype=float)), errors="coerce")

    go = test[test["condition"] == "go"]
    stop = test[test["condition"] == "stop"]

    go_correct_rt = go[(go["correct_trial"] == 1) & go["rt"].notna()]["rt"]
    go_all_responded_rt = go[go["rt"].notna()]["rt"]
    stop_failed = stop[(stop["correct_trial"] == 0) & stop["rt"].notna()]
    stop_ssd = stop["SSD"].dropna()

    results = {
        "task": "Stop Signal (ExpFactory)",
        "format": "expfactory_stop_signal",
        "n_total": len(test),
        "summary": {
            "go_accuracy": go["correct_trial"].mean() if len(go) else np.nan,
            "go_omission_rate": go["rt"].isna().mean() if len(go) else np.nan,
            "go_rt": go_correct_rt.mean() if len(go_correct_rt) else np.nan,
            "go_rt_all_responses": go_all_responded_rt.mean() if len(go_all_responded_rt) else np.nan,
            "mean_stop_failure_RT": stop_failed["rt"].mean() if len(stop_failed) else np.nan,
            "stop_accuracy": stop["correct_trial"].mean() if len(stop) else np.nan,
            "min_SSD": stop_ssd.min() if len(stop_ssd) else np.nan,
            "mean_SSD": stop_ssd.mean() if len(stop_ssd) else np.nan,
            "max_SSD": stop_ssd.max() if len(stop_ssd) else np.nan,
            "final_SSD": stop["SSD"].iloc[-1] if len(stop) else np.nan,
        },
    }
    return results


def analyze_cognitionrun_stroop(df: pd.DataFrame) -> dict:
    """Analyze Cognition.run Stroop data."""
    trials = df[(df["text"].notna()) & (df["text"] != "") & (df["colour"].notna()) & (df["colour"] != "")].copy()
    trials["rt"] = pd.to_numeric(trials["rt"], errors="coerce")
    trials["condition"] = np.where(trials["text"] == trials["colour"], "congruent", "incongruent")
    key_map = {"red": "r", "green": "g", "blue": "b", "yellow": "y"}
    trials["correct_key"] = trials["colour"].map(key_map)
    trials["correct"] = trials["response"] == trials["correct_key"]
    trials["omission"] = trials["response"].isna() | (trials["rt"].isna())

    results = {"task": "Stroop (Cognition.run)", "format": "cognitionrun_stroop", "n_total": len(trials)}
    summary = {}
    for cond in ["congruent", "incongruent"]:
        g = trials[trials["condition"] == cond]
        correct = g[g["correct"]]
        rt_correct = correct["rt"].dropna()
        summary[f"{cond}_accuracy"] = g["correct"].mean() if len(g) else np.nan
        summary[f"{cond}_omission_rate"] = g["omission"].mean() if len(g) else np.nan
        summary[f"{cond}_rt"] = rt_correct.mean() if len(rt_correct) else np.nan
    results["summary"] = summary
    return results


def analyze_stopit(df: pd.DataFrame) -> dict:
    """Analyze STOP-IT Stop Signal data."""
    df = df.copy()
    df["rt"] = pd.to_numeric(df["rt"], errors="coerce")
    df["correct"] = df["correct"].astype(str).str.lower() == "true"
    df["block_i"] = pd.to_numeric(df["block_i"], errors="coerce")
    df["SSD"] = pd.to_numeric(df["SSD"], errors="coerce")

    exp = df[df["block_i"] > 0]  # exclude practice block 0
    go = exp[exp["signal"] == "no"]
    stop = exp[exp["signal"] == "yes"]

    go_omission = go["rt"].isna() | (go["response"] == "undefined")
    go_correct_rt = go[go["correct"] & ~go_omission]["rt"].dropna()
    go_all_responded_rt = go[~go_omission]["rt"].dropna()
    stop_failed = stop[~stop["correct"]]
    stop_failed_rt = stop_failed["rt"].dropna()
    stop_ssd = stop["SSD"].dropna()

    results = {
        "task": "Stop Signal (STOP-IT)",
        "format": "stopit",
        "n_total": len(exp),
        "summary": {
            "go_accuracy": go["correct"].mean() if len(go) else np.nan,
            "go_omission_rate": go_omission.mean() if len(go) else np.nan,
            "go_rt": go_correct_rt.mean() if len(go_correct_rt) else np.nan,
            "go_rt_all_responses": go_all_responded_rt.mean() if len(go_all_responded_rt) else np.nan,
            "mean_stop_failure_RT": stop_failed_rt.mean() if len(stop_failed_rt) else np.nan,
            "stop_accuracy": stop["correct"].mean() if len(stop) else np.nan,
            "min_SSD": stop_ssd.min() if len(stop_ssd) else np.nan,
            "mean_SSD": stop_ssd.mean() if len(stop_ssd) else np.nan,
            "max_SSD": stop_ssd.max() if len(stop_ssd) else np.nan,
            "final_SSD": stop["SSD"].iloc[-1] if len(stop) else np.nan,
        },
    }
    return results


ANALYZERS = {
    "expfactory_stroop": analyze_expfactory_stroop,
    "expfactory_stop_signal": analyze_expfactory_stop_signal,
    "cognitionrun_stroop": analyze_cognitionrun_stroop,
    "stopit": analyze_stopit,
}

print("Defined analyzers for:", list(ANALYZERS.keys()))

Defined analyzers for: ['expfactory_stroop', 'expfactory_stop_signal', 'cognitionrun_stroop', 'stopit']


In [59]:
# Analyze all data files
all_results = []
for f in datafiles:
    df = pd.read_csv(f)
    fmt = detect_format(df)
    label = f"{f.parent.parent.name}/{f.parent.name}"

    if fmt in ANALYZERS:
        results = ANALYZERS[fmt](df)
        results["file"] = label
        all_results.append(results)
    else:
        print(f"--- {label}: format={fmt} (no analyzer), {len(df)} rows ---")

print(f"Analyzed {len(all_results)} files")

Analyzed 120 files


In [60]:
STOP_SIGNAL_COLS = [
    "task", "run", "go_accuracy", "go_omission_rate", "go_rt", "go_rt_all_responses",
    "mean_stop_failure_RT", "stop_accuracy", "max_SSD", "mean_SSD", "min_SSD", "final_SSD",
]
STROOP_COLS = [
    "task", "run", "congruent_accuracy", "congruent_omission_rate", "congruent_rt",
    "incongruent_accuracy", "incongruent_omission_rate", "incongruent_rt",
]

stop_rows, stroop_rows = [], []
for r in all_results:
    row = {"task": r["task"], "run": r["file"].split("/")[-1], **r["summary"]}
    if r["format"] in ("expfactory_stop_signal", "stopit"):
        stop_rows.append(row)
    else:
        stroop_rows.append(row)

if stop_rows:
    stop_df = pd.DataFrame(stop_rows, columns=STOP_SIGNAL_COLS)
    print("Stop Signal Summary\n")
    print(stop_df.to_string(index=False, float_format=lambda x: f"{x:.3f}" if abs(x) < 2 else f"{x:.1f}"))
    stop_df.to_csv(output_dir / "stop_signal_summary.csv", index=False)
    print(f"\nSaved to {output_dir / 'stop_signal_summary.csv'}")

if stroop_rows:
    stroop_df = pd.DataFrame(stroop_rows, columns=STROOP_COLS)
    print("\n\nStroop Summary\n")
    print(stroop_df.to_string(index=False, float_format=lambda x: f"{x:.3f}" if abs(x) < 2 else f"{x:.1f}"))
    stroop_df.to_csv(output_dir / "stroop_summary.csv", index=False)
    print(f"\nSaved to {output_dir / 'stroop_summary.csv'}")

Stop Signal Summary

                    task                 run  go_accuracy  go_omission_rate  go_rt  go_rt_all_responses  mean_stop_failure_RT  stop_accuracy  max_SSD  mean_SSD  min_SSD  final_SSD
Stop Signal (ExpFactory) 2026-03-01_12-54-37        0.942             0.033  555.9                559.4                 503.5          0.533    600.0     234.2    0.000      500.0
Stop Signal (ExpFactory) 2026-03-01_12-54-39        0.925             0.050  563.1                564.3                 493.7          0.517    600.0     446.7    200.0      400.0
Stop Signal (ExpFactory) 2026-03-01_12-54-41        0.950             0.050  653.5                653.5                 564.9          0.533    650.0     420.0    200.0      500.0
Stop Signal (ExpFactory) 2026-03-01_12-54-43        0.875             0.075  635.8                640.1                 532.7          0.567    650.0     445.0    150.0      600.0
Stop Signal (ExpFactory) 2026-03-01_12-54-45        0.908             0.050  63